In [1]:
import torch
import torch.nn as nn
import torch.quantization
import torchvision
from torchvision.models import mobilenet_v2
import torch_pruning as tp
import os
import time

In [2]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Baseline model loaded.")

Baseline model loaded.


In [3]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Model quantized.")

C:\Users\user\AppData\Local\Temp\ipykernel_18188\3575092245.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Model quantized.


In [4]:
example_inputs = torch.randn(1, 3, 224, 224)

importance = tp.importance.MagnitudeImportance(p=2)

ignored_layers = [quantized_model.classifier]

pruner = tp.pruner.MagnitudePruner(
    quantized_model,
    example_inputs,
    importance=importance,
    pruning_ratio=0.2,
    ignored_layers=ignored_layers
)

print("Pruner created successfully.")

Pruner created successfully.


In [5]:
pruner.step()

print("Structural pruning applied successfully.")

Structural pruning applied successfully.


In [6]:
total_params = sum(p.numel() for p in quantized_model.parameters())

print(f"Total parameters after structural pruning: {total_params:,}")

Total parameters after structural pruning: 1,435,140


In [7]:
torch.save(
    quantized_model.state_dict(),
    "../models/structural_quantize_then_prune_model.pth"
)

size_mb = os.path.getsize(
    "../models/structural_quantize_then_prune_model.pth"
) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 6.91 MB


In [8]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up
    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    return (end - start) / runs

In [9]:
input_tensor = torch.randn(1, 3, 224, 224)

cpu_latency = benchmark(
    quantized_model,
    "cpu",
    input_tensor
)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

RuntimeError: could not create a primitive descriptor for the matmul primitive. Run workload with environment variable ONEDNN_VERBOSE=all to get additional diagnostic information.

In [10]:
print(quantized_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): DynamicQuantizedLinear(in_features=1280, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)


In [11]:
print(sum(p.numel() for p in quantized_model.parameters()))

1435140


In [12]:
for name, module in quantized_model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(name, module.out_channels)

features.0.0 25
features.1.conv.0.0 25
features.1.conv.1 12
features.2.conv.0.0 76
features.2.conv.1.0 76
features.2.conv.2 19
features.3.conv.0.0 115
features.3.conv.1.0 115
features.3.conv.2 19
features.4.conv.0.0 115
features.4.conv.1.0 115
features.4.conv.2 25
features.5.conv.0.0 153
features.5.conv.1.0 153
features.5.conv.2 25
features.6.conv.0.0 153
features.6.conv.1.0 153
features.6.conv.2 25
features.7.conv.0.0 153
features.7.conv.1.0 153
features.7.conv.2 51
features.8.conv.0.0 307
features.8.conv.1.0 307
features.8.conv.2 51
features.9.conv.0.0 307
features.9.conv.1.0 307
features.9.conv.2 51
features.10.conv.0.0 307
features.10.conv.1.0 307
features.10.conv.2 51
features.11.conv.0.0 307
features.11.conv.1.0 307
features.11.conv.2 76
features.12.conv.0.0 460
features.12.conv.1.0 460
features.12.conv.2 76
features.13.conv.0.0 460
features.13.conv.1.0 460
features.13.conv.2 76
features.14.conv.0.0 460
features.14.conv.1.0 460
features.14.conv.2 128
features.15.conv.0.0 768
feat

In [13]:
print(type(quantized_model.classifier[1]))

<class 'torch.ao.nn.quantized.dynamic.modules.linear.Linear'>


In [14]:
print(quantized_model.classifier[1])

DynamicQuantizedLinear(in_features=1280, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)


In [16]:
with torch.no_grad():
    x = torch.randn(1, 3, 224, 224)

    features = quantized_model.features(x)

    print(features.shape)

torch.Size([1, 1024, 7, 7])


Manually reconstruct the classifier.

In [18]:
print(quantized_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): DynamicQuantizedLinear(in_features=1280, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)


In [19]:
import torch.nn as nn

quantized_model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(1024, 1000)
)

print(quantized_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1024, out_features=1000, bias=True)
)


In [20]:
with torch.no_grad():
    x = torch.randn(1,3,224,224)
    y = quantized_model(x)

print(y.shape)

torch.Size([1, 1000])


In [21]:
torch.save(
    quantized_model.state_dict(),
    "../models/structural_quantize_then_prune_repaired.pth"
)

import os

size_mb = os.path.getsize(
    "../models/structural_quantize_then_prune_repaired.pth"
) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 9.60 MB


In [22]:
input_tensor = torch.randn(1,3,224,224)

cpu_latency = benchmark(
    quantized_model,
    "cpu",
    input_tensor
)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.058864 seconds
